In [1]:
import collections
import random
from random import uniform
from math import *
import numpy as np
import scipy.optimize as opt
from scipy.optimize import minimize
import concurrent.futures
import warnings



def H(c):
    """
    Entropy function
    """
    if c == 0. or c == 1.:
        return 0.
    
    if c < 0. or c > 1.:
        return -1000
    
    return -(c * log2(c) + (1 - c) * log2(1 - c))

def binomH(n, k):
    """
    Asymptotic exponent of binomial coefficient.
    """
    if n <= 0:
        return 0.0

    return n * H(k / n)


def multiH(n, c):
    """
    Asymptotic exponent of multinomial coefficient.
    """
    if sum(c)>n:
        return 0
    tot=0
    val=n
    for i in c:
        tot+=binomH(n,i)
        n-=i
    return tot


def wrap(f,g) :
    def inner(x):
        return f(g(*x))
    return inner

def r(x,y,z):
    return [(ru(x,y)) for i in range(z)]


### Improved Nested

In [2]:
set_vars = collections.namedtuple('LWE', ' o1_0 z1_0 z2_0 o1_1 z1_1 z2_1 t_1 g b lmd ')

num_params = 10

def lwe(f) : return wrap(f, set_vars)


# ============================================================================
# Level-0 Parameters
# ============================================================================

n1_0 = lambda x : x.b * x.g * w/2
n0_0 = lambda x : x.g - x.b*x.g*w


# ============================================================================
# Level-1 Parameters
# ============================================================================

n2_1 = lambda x : x.o1_0 + x.z2_0             
n1_1 = lambda x : n1_0(x)/2 + x.z1_0           
n0_1 = lambda x : n1_0(x) + n0_0(x) - 2*(x.z1_0 + x.z2_0 +x.o1_0) 


### Layer-1 representation count exponent.
R1 =  lambda x : 2*multiH(n1_0(x),[x.o1_0, x.o1_0, n1_0(x)/2 - x.o1_0]) + multiH(n0_0(x), [x.z1_0, x.z2_0,x.z1_0, x.z2_0])







# ============================================================================
# Level-2 Parameters
# ============================================================================

n2_2 = lambda x : x.o1_1 + x.z2_1 + x.t_1            
n1_2 = lambda x : n1_1(x)/2 + n2_1(x) + x.z1_1 - 2*x.t_1           
n0_2 = lambda x : n0_1(x) + n1_1(x) - 2*(x.z1_1 + x.z2_1 +x.o1_1 - x.t_1)



###  Layer-2 representation count exponent.
R2 =  lambda x : 2*multiH(n1_1(x),[x.o1_1, x.o1_1, n1_1(x)/2 - x.o1_1]) + multiH(n0_1(x), [x.z1_1, x.z2_1,x.z1_1, x.z2_1]) + 2*multiH(n2_1(x),[x.t_1, x.t_1])





# ============================================================================
# Domain Size Exponents
# ============================================================================


###  Exponent corresponding to the lower-level search domain.
domain = lambda x : multiH(x.g,[n2_2(x), n2_2(x), n1_2(x), n1_2(x)])+ multiH((1-x.g)/4, [(1-x.b*x.g)*w/8,(1-x.b*x.g)*w/8])



### Exponent corresponding to the restricted secret domain.
ss = lambda x : multiH(x.g,[n1_0(x) , n1_0(x)]) + 4*multiH((1-x.g)/4,[(1-x.g*x.b)*w/8,(1-x.g*x.b)*w/8])



# ============================================================================
# Objective Function
# ============================================================================

def time(x):

    x = set_vars(*x)

    good_D = multiH(1, [w / 2, w / 2]) - ss(x)

    good_R = max(0, domain(x) - R1(x))

    collisions_when_R_good = max(0, domain(x) - 2 * R2(x))

    one_collision = domain(x)

    return (
        good_R
        + collisions_when_R_good
        + one_collision
        + good_D
    )


In [3]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("Optimized_data_Improved-Nested-2star.csv")

# Extract rows excluding the header as tuples
rows_as_tuples = [tuple(row) for row in df.itertuples(index=False, name=None)]

# Print the tuples
lst1=[]
for t in rows_as_tuples:
    w=t[0]
    # w1=int(w*100)
    expt = t[1]
    lst = list(t[2:])
    # if 0.16<=w<=0.48 and w1%2==0:
    x=set_vars(*lst)
    lst1.append((w,round(expt,4)))
    if round(time(x),4)!=round(expt,4):
        print(w, time(x), expt)

In [4]:
len(lst1)

48

### Shifted 3

In [5]:
set_vars = collections.namedtuple('LWE', ' thm_1 tm1_1 tm0_1 om1_1 om2_1 th_1 t1_1   o1_0 o2_0 z1_0 z2_0 z3_0 t1_0 t0_0 t0_1 o1_1 o2_1 z1_1 z2_1 z3_1 g b lmd')

num_params = 23


def lwe(f) : return wrap(f, set_vars)


# ============================================================================
# Level-0 Parameters
# ============================================================================

n0_0 = lambda x: x.b * x.g * w / 2
n1_0 = lambda x: x.g - x.b * x.g * w
n2_0 = lambda x: x.b * x.g * w / 2


# ============================================================================
# Level-1 Parameters
# ============================================================================

n_3_1 = lambda x : x.z3_0
n_2_1 = lambda x : x.z2_0 + x.o2_0
n_1_1 = lambda x : x.z1_0 + x.o1_0 + x.t1_0 
n0_1 = lambda x : n1_0(x)/2 - (x.o1_0 + x.o2_0) + n0_0(x) - 2*(x.z1_0 + x.z2_0 + x.z3_0) + x.t0_0
n1_1 = lambda x : x.z1_0 + n1_0(x)/2 - (x.o1_0 + x.o2_0) + n2_0(x) - 2*(x.t0_0 + x.t1_0)
n2_1 = lambda x : x.z2_0 + x.o1_0 + x.t0_0 
n3_1 = lambda x : x.z3_0 + x.o2_0 + x.t1_0 


### Layer-1 representation count exponent.
R1 = lambda x : multiH(n0_0(x), [x.z3_0, x.z2_0, x.z1_0, x.z3_0, x.z2_0, x.z1_0]) + multiH(n1_0(x), [x.o2_0, x.o1_0, x.o2_0, x.o1_0, n1_0(x)/2 - x.o1_0 - x.o2_0]) + multiH(n2_0(x),[x.t1_0, x.t0_0, x.t1_0, x.t0_0])







# ============================================================================
# Level-2 Parameters
# ============================================================================

n_3_2 = lambda x : x.thm_1 + x.tm1_1 + x.om2_1 + x.z3_1
n_2_2 = lambda x : n_3_1(x)/2 - x.thm_1 + x.tm0_1 + x.om1_1 + x.z2_1 + x.o2_1
n_1_2 = lambda x : n_3_1(x)/2 - x.thm_1 + n_2_1(x) - 2*(x.tm0_1+x.tm1_1) + n_1_1(x)/2 - x.om1_1 - x.om2_1 + x.z1_1 + x.o1_1 + x.t1_1 
n0_2 =  lambda x : x.thm_1 + x.tm0_1 + n_1_1(x)/2 - x.om1_1 - x.om2_1 + n0_1(x) - 2*(x.z1_1+x.z2_1+x.z3_1) + n1_1(x)/2 - x.o1_1 - x.o2_1 + x.t0_1 + x.th_1
n1_2 =  lambda x : x.tm1_1 + x.om1_1 + x.z1_1 + n1_1(x)/2 - x.o1_1 - x.o2_1 + n2_1(x) - 2*(x.t0_1+x.t1_1) + n3_1(x)/2 - x.th_1
n2_2 =  lambda x : x.om2_1 + x.z2_1 + x.o1_1 + x.t0_1 + n3_1(x)/2 - x.th_1
n3_2 =  lambda x : x.z3_1 + x.o2_1 + x.t1_1 + x.th_1



###  Layer-2 representation count exponent.
R2 = lambda x : multiH(n_3_1(x), [x.thm_1, x.thm_1, n_3_1(x)/2-x.thm_1]) + multiH(n_2_1(x), [x.tm1_1, x.tm0_1, x.tm1_1, x.tm0_1]) + multiH(n_1_1(x), [x.om2_1, x.om1_1, x.om1_1, x.om2_1, n_1_1(x)/2 - x.om1_1 - x.om2_1]) + multiH(n0_1(x), [x.z3_1, x.z2_1, x.z1_1,x.z3_1, x.z2_1, x.z1_1]) + multiH(n1_1(x), [x.o2_1, x.o1_1, x.o1_1, x.o2_1, n1_1(x)/2 - x.o1_1 - x.o2_1]) + multiH(n2_1(x), [x.t1_1, x.t0_1, x.t1_1, x.t0_1]) + multiH(n3_1(x), [x.th_1, x.th_1, n3_1(x)/2-x.th_1])





# ============================================================================
# Domain Size Exponents
# ============================================================================


###  Exponent corresponding to the lower-level search domain.
domain = lambda x : multiH(x.g,[n3_2(x), n_3_2(x), n2_2(x), n_2_2(x), n1_2(x), n_1_2(x)])+ multiH((1-x.g)/4, [(1-x.b*x.g)*w/8,(1-x.b*x.g)*w/8])



### Exponent corresponding to the restricted secret domain.
ss = lambda x : multiH(x.g,[x.b*x.g*w/2 , x.b*x.g*w/2]) + 4*multiH((1-x.g)/4,[(1-x.g*x.b)*w/8,(1-x.g*x.b)*w/8])



# ============================================================================
# Objective Function
# ============================================================================

def time(x):

    x = set_vars(*x)

    good_D = multiH(1, [w / 2, w / 2]) - ss(x)

    good_R = max(0, domain(x) - R1(x))

    collisions_when_R_good = max(0, domain(x) - 2 * R2(x))

    one_collision = domain(x)

    return (
        good_R
        + collisions_when_R_good
        + one_collision
        + good_D
    )


In [6]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("Optimized_data_Shifted-3.csv")

# Extract rows excluding the header as tuples
rows_as_tuples = [tuple(row) for row in df.itertuples(index=False, name=None)]

# Print the tuples
lst1=[]
for t in rows_as_tuples:
    w=t[0]
    # w1=int(w*100)
    expt = t[1]
    lst = list(t[2:])
    # if 0.16<=w<=0.48 and w1%2==0:
    x=set_vars(*lst)
    lst1.append((w,round(expt,4)))
    if round(time(x),4)!=round(expt,4):
        print(w, time(x), expt)

In [7]:
len(lst1)

42

### Shifted-3+ISD

In [13]:
set_vars = collections.namedtuple('LWE', ' thm_1 tm1_1 tm0_1 om1_1 om2_1 th_1 t1_1   o1_0 o2_0 z1_0 z2_0 z3_0 t1_0 t0_0 t0_1 o1_1 o2_1 z1_1 z2_1 z3_1 g b delta lmd')

num_params = 23


def lwe(f) : return wrap(f, set_vars)



# ============================================================================
# Level-0 Parameters
# ============================================================================

w = lambda x : w1-x.delta

n0_0 = lambda x : x.b * x.g * w(x)/2
n1_0 = lambda x : x.g - x.b*x.g*w(x)
n2_0 = lambda x : x.b * x.g * w(x)/2


# ============================================================================
# Level-1 Parameters
# ============================================================================

n_3_1 = lambda x : x.z3_0
n_2_1 = lambda x : x.z2_0 + x.o2_0
n_1_1 = lambda x : x.z1_0 + x.o1_0 + x.t1_0 
n0_1 = lambda x : n1_0(x)/2 - (x.o1_0 + x.o2_0) + n0_0(x) - 2*(x.z1_0 + x.z2_0 + x.z3_0) + x.t0_0
n1_1 = lambda x : x.z1_0 + n1_0(x)/2 - (x.o1_0 + x.o2_0) + n2_0(x) - 2*(x.t0_0 + x.t1_0)
n2_1 = lambda x : x.z2_0 + x.o1_0 + x.t0_0 
n3_1 = lambda x : x.z3_0 + x.o2_0 + x.t1_0 


### Layer-1 representation count exponent.
R1 = lambda x : multiH(n0_0(x), [x.z3_0, x.z2_0, x.z1_0, x.z3_0, x.z2_0, x.z1_0]) + multiH(n1_0(x), [x.o2_0, x.o1_0, x.o2_0, x.o1_0, n1_0(x)/2 - x.o1_0 - x.o2_0]) + multiH(n2_0(x),[x.t1_0, x.t0_0, x.t1_0, x.t0_0])







# ============================================================================
# Level-2 Parameters
# ============================================================================

n_3_2 = lambda x : x.thm_1 + x.tm1_1 + x.om2_1 + x.z3_1
n_2_2 = lambda x : n_3_1(x)/2 - x.thm_1 + x.tm0_1 + x.om1_1 + x.z2_1 + x.o2_1
n_1_2 = lambda x : n_3_1(x)/2 - x.thm_1 + n_2_1(x) - 2*(x.tm0_1+x.tm1_1) + n_1_1(x)/2 - x.om1_1 - x.om2_1 + x.z1_1 + x.o1_1 + x.t1_1 
n0_2 =  lambda x : x.thm_1 + x.tm0_1 + n_1_1(x)/2 - x.om1_1 - x.om2_1 + n0_1(x) - 2*(x.z1_1+x.z2_1+x.z3_1) + n1_1(x)/2 - x.o1_1 - x.o2_1 + x.t0_1 + x.th_1
n1_2 =  lambda x : x.tm1_1 + x.om1_1 + x.z1_1 + n1_1(x)/2 - x.o1_1 - x.o2_1 + n2_1(x) - 2*(x.t0_1+x.t1_1) + n3_1(x)/2 - x.th_1
n2_2 =  lambda x : x.om2_1 + x.z2_1 + x.o1_1 + x.t0_1 + n3_1(x)/2 - x.th_1
n3_2 =  lambda x : x.z3_1 + x.o2_1 + x.t1_1 + x.th_1


###  Layer-2 representation count exponent.
R2 = lambda x : multiH(n_3_1(x), [x.thm_1, x.thm_1, n_3_1(x)/2-x.thm_1]) + multiH(n_2_1(x), [x.tm1_1, x.tm0_1, x.tm1_1, x.tm0_1]) + multiH(n_1_1(x), [x.om2_1, x.om1_1, x.om1_1, x.om2_1, n_1_1(x)/2 - x.om1_1 - x.om2_1]) + multiH(n0_1(x), [x.z3_1, x.z2_1, x.z1_1,x.z3_1, x.z2_1, x.z1_1]) + multiH(n1_1(x), [x.o2_1, x.o1_1, x.o1_1, x.o2_1, n1_1(x)/2 - x.o1_1 - x.o2_1]) + multiH(n2_1(x), [x.t1_1, x.t0_1, x.t1_1, x.t0_1]) + multiH(n3_1(x), [x.th_1, x.th_1, n3_1(x)/2-x.th_1])





# ============================================================================
# Domain Size Exponents
# ============================================================================


###  Exponent corresponding to the lower-level search domain.
domain = lambda x : multiH(x.g,[n3_2(x), n_3_2(x), n2_2(x), n_2_2(x), n1_2(x), n_1_2(x)])+ multiH((1-x.g)/4, [(1-x.b*x.g)*w(x)/8,(1-x.b*x.g)*w(x)/8])



### Exponent corresponding to the restricted secret domain.
ss = lambda x : multiH(x.g,[x.b*x.g*w(x)/2 , x.b*x.g*w(x)/2]) + 4*multiH((1-x.g)/4,[(1-x.g*x.b)*w(x)/8,(1-x.g*x.b)*w(x)/8])



# ============================================================================
# Objective Function
# ============================================================================

def time(x):

    x = set_vars(*x)
    
    if x.delta < 0.000001:
        ISD =0
    else:
        ISD = multiH(2,[1/3 + w1/2, 1/3 + w1/2]) - multiH(1,[w(x)/2, w(x)/2]) - multiH(1,[1/3 + x.delta/2, 1/3 + x.delta/2])

    good_D = multiH(1, [w(x) / 2, w(x) / 2]) - ss(x)

    good_R = max(0, domain(x) - R1(x))

    collisions_when_R_good = max(0, domain(x) - 2 * R2(x))

    one_collision = domain(x)

    return (
        ISD
        + good_R
        + collisions_when_R_good
        + one_collision
        + good_D
    )


In [15]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("Optimized_data_ISD+Shifted.csv")

# Extract rows excluding the header as tuples
rows_as_tuples = [tuple(row) for row in df.itertuples(index=False, name=None)]

# Print the tuples
lst1=[]
for t in rows_as_tuples:
    w1=t[0]
    # w1=int(w*100)
    expt = t[1]
    lst = list(t[2:])
    # if 0.16<=w<=0.48 and w1%2==0:
    x=set_vars(*lst)
    lst1.append((w1,round(expt,4)))
    print(time(x))
    if round(time(x),4)!=round(expt,4):
        print(w1, time(x), expt)

0.8605795495593884
0.8607841083850147


### Hybrid

In [16]:
set_vars = collections.namedtuple('LWE', 'bo1_0 bz1_0 bz2_0 bo1_1 bz1_1 bz2_1 bt_1 thm_1 tm1_1 tm0_1 om1_1 om2_1 th_1 t1_1  o1_0 o2_0 z1_0 z2_0 z3_0 t1_0 t0_0 t0_1 o1_1 o2_1 z1_1 z2_1 z3_1 g1 g2 b1 b2 lmd')

num_params = 32

def lwe(f) : return wrap(f, set_vars)


# =================================================================================================================
#                                                  Shifted   
# =================================================================================================================

# ========================================================
# Level-0 Parameters
# ========================================================

n0_0 = lambda x : x.b1 * x.g1 * w/2
n1_0 = lambda x : x.g1 - x.b1*x.g1*w
n2_0 = lambda x : x.b1 * x.g1 * w/2


# ========================================================
# Level-1 Parameters
# ========================================================

n_3_1 = lambda x : x.z3_0
n_2_1 = lambda x : x.z2_0 + x.o2_0
n_1_1 = lambda x : x.z1_0 + x.o1_0 + x.t1_0 
n0_1 = lambda x : n1_0(x)/2 - (x.o1_0 + x.o2_0) + n0_0(x) - 2*(x.z1_0 + x.z2_0 + x.z3_0) + x.t0_0
n1_1 = lambda x : x.z1_0 + n1_0(x)/2 - (x.o1_0 + x.o2_0) + n2_0(x) - 2*(x.t0_0 + x.t1_0)
n2_1 = lambda x : x.z2_0 + x.o1_0 + x.t0_0 
n3_1 = lambda x : x.z3_0 + x.o2_0 + x.t1_0 


### Layer-1 representation count exponent due to Shifted part.
R1 = lambda x : multiH(n0_0(x), [x.z3_0, x.z2_0, x.z1_0, x.z3_0, x.z2_0, x.z1_0]) + multiH(n1_0(x), [x.o2_0, x.o1_0, x.o2_0, x.o1_0, n1_0(x)/2 - x.o1_0 - x.o2_0]) + multiH(n2_0(x),[x.t1_0, x.t0_0, x.t1_0, x.t0_0])



# =======================================================
# Level-2 Parameters
# =======================================================

n_3_2 = lambda x : x.thm_1 + x.tm1_1 + x.om2_1 + x.z3_1
n_2_2 = lambda x : n_3_1(x)/2 - x.thm_1 + x.tm0_1 + x.om1_1 + x.z2_1 + x.o2_1
n_1_2 = lambda x : n_3_1(x)/2 - x.thm_1 + n_2_1(x) - 2*(x.tm0_1+x.tm1_1) + n_1_1(x)/2 - x.om1_1 - x.om2_1 + x.z1_1 + x.o1_1 + x.t1_1 
n0_2 =  lambda x : x.thm_1 + x.tm0_1 + n_1_1(x)/2 - x.om1_1 - x.om2_1 + n0_1(x) - 2*(x.z1_1+x.z2_1+x.z3_1) + n1_1(x)/2 - x.o1_1 - x.o2_1 + x.t0_1 + x.th_1
n1_2 =  lambda x : x.tm1_1 + x.om1_1 + x.z1_1 + n1_1(x)/2 - x.o1_1 - x.o2_1 + n2_1(x) - 2*(x.t0_1+x.t1_1) + n3_1(x)/2 - x.th_1
n2_2 =  lambda x : x.om2_1 + x.z2_1 + x.o1_1 + x.t0_1 + n3_1(x)/2 - x.th_1
n3_2 =  lambda x : x.z3_1 + x.o2_1 + x.t1_1 + x.th_1



###  Layer-2 representation count exponent due to shifted part.
R2 = lambda x : multiH(n_3_1(x), [x.thm_1, x.thm_1, n_3_1(x)/2-x.thm_1]) + multiH(n_2_1(x), [x.tm1_1, x.tm0_1, x.tm1_1, x.tm0_1]) + multiH(n_1_1(x), [x.om2_1, x.om1_1, x.om1_1, x.om2_1, n_1_1(x)/2 - x.om1_1 - x.om2_1]) + multiH(n0_1(x), [x.z3_1, x.z2_1, x.z1_1,x.z3_1, x.z2_1, x.z1_1]) + multiH(n1_1(x), [x.o2_1, x.o1_1, x.o1_1, x.o2_1, n1_1(x)/2 - x.o1_1 - x.o2_1]) + multiH(n2_1(x), [x.t1_1, x.t0_1, x.t1_1, x.t0_1]) + multiH(n3_1(x), [x.th_1, x.th_1, n3_1(x)/2-x.th_1])

# =================================================================================================================
#                                                  Improved Nested-2star   
# =================================================================================================================

# ========================================================
# Level-0 Parameters
# ========================================================

bn1_0 = lambda x : x.b2 * x.g2 * w/2
bn0_0 = lambda x : x.g2 - x.b2*x.g2*w


# ========================================================
# Level-1 Parameters
# ========================================================

bn2_1 = lambda x : x.bo1_0 + x.bz2_0             
bn1_1 = lambda x : bn1_0(x)/2 + x.bz1_0           
bn0_1 = lambda x : bn1_0(x) + bn0_0(x) - 2*(x.bz1_0 + x.bz2_0 +x.bo1_0)  


### Layer-1 representation count exponent due to Nested-2star part.
bR_1 =  lambda x : 2*multiH(bn1_0(x),[x.bo1_0, x.bo1_0, bn1_0(x)/2 - x.bo1_0]) + multiH(bn0_0(x), [x.bz1_0, x.bz2_0,x.bz1_0, x.bz2_0])



# =======================================================
# Level-2 Parameters
# =======================================================

bn2_2 = lambda x : x.bo1_1 + x.bz2_1 + x.bt_1            
bn1_2 = lambda x : bn1_1(x)/2 + bn2_1(x) + x.bz1_1 - 2*x.bt_1           
bn0_2 = lambda x : bn0_1(x) + bn1_1(x) - 2*(x.bz1_1 + x.bz2_1 +x.bo1_1 - x.bt_1)



###  Layer-2 representation count exponent due to Nested-2star part.
bR_2 =  lambda x : 2*multiH(bn1_1(x),[x.bo1_1, x.bo1_1, bn1_1(x)/2 - x.bo1_1]) + multiH(bn0_1(x), [x.bz1_1, x.bz2_1,x.bz1_1, x.bz2_1]) + 2*multiH(bn2_1(x),[x.bt_1, x.bt_1])




# ============================================================================
# Total representation exponents
# ============================================================================

R1_hyb = lambda x : R1(x) + bR_1(x)
R2_hyb = lambda x : R2(x) + bR_2(x)


# ============================================================================
# Domain Size Exponents
# ============================================================================


###  Exponent corresponding to the lower-level search domain.
domain = lambda x : multiH(x.g1,[n3_2(x), n_3_2(x), n2_2(x), n_2_2(x), n1_2(x), n_1_2(x)])+ multiH(x.g2,[bn2_2(x), bn2_2(x), bn1_2(x), bn1_2(x)])+ multiH((1-x.g1-x.g2)/4, [(1-x.b1*x.g1-x.b2*x.g2)*w/8,(1-x.b1*x.g1-x.b2*x.g2)*w/8])



### Exponent corresponding to the restricted secret domain.
ss = lambda x : multiH(x.g1,[x.b1*x.g1*w/2 , x.b1*x.g1*w/2]) +multiH(x.g2,[x.b2*x.g2*w/2 , x.b2*x.g2*w/2]) + 4*multiH((1-x.g1-x.g2)/4, [(1-x.b1*x.g1-x.b2*x.g2)*w/8,(1-x.b1*x.g1-x.b2*x.g2)*w/8])



# ============================================================================
# Objective Function
# ============================================================================

def time(x):

    x = set_vars(*x)

    good_D = multiH(1, [w / 2, w / 2]) - ss(x)

    good_R = max(0, domain(x) - R1_hyb(x))

    collisions_when_R_good = max(0, domain(x) - 2 * R2_hyb(x))

    one_collision = domain(x)

    return (
        good_R
        + collisions_when_R_good
        + one_collision
        + good_D
    )



In [18]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("Optimized_data_Tab-1_Hybrid.csv")

# Extract rows excluding the header as tuples
rows_as_tuples = [tuple(row) for row in df.itertuples(index=False, name=None)]

# Print the tuples
lst1=[]
for t in rows_as_tuples:
    w=t[0]
    # w1=int(w*100)
    expt = t[1]
    lst = list(t[2:])
    # if 0.16<=w<=0.48 and w1%2==0:
    x=set_vars(*lst)
    lst1.append((w,round(expt,4)))
    if round(time(x),4)!=round(expt,4):
        print(w, time(x), expt)

### 

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv("Optimized_data_Tab-1_Hybrid.csv")

# Extract rows excluding the header as tuples
rows_as_tuples = [tuple(row) for row in df.itertuples(index=False, name=None)]

# Print the tuples
lst1=[]
for t in rows_as_tuples:
    w=t[0]
    # w1=int(w*100)
    expt = t[1]
    lst = list(t[2:])
    # if 0.16<=w<=0.48 and w1%2==0:
    x=set_vars(*lst)
    lst1.append((w,round(expt,4)))
    if round(time(x),4)!=round(expt,4):
        print(w, time(x), expt)